In [1]:
# !pip install jsonschema
import json
from jsonschema import validate
from constants import ValidationError, ErrorType, assert_all_logs
from engine import Engine
import os

data_file = 'data.json'
schema_file = 'schema.json'
data_enum_error_file = 'data_enum_error.json'
data_type_error_file = 'data_type_error.json'
data_no_schema_error_file = 'data_no_schema_error.json'
data_unclosed_error_file = 'data_unclosed_error.json'

In [2]:
json_data = json.load(open(data_file))
json_schema = json.load(open(schema_file))

# Validate the JSON data against the schema using jsonschema library
# jsonschema requires valid python dict for both data and schema
validate(instance=json_data, schema=json_schema)

In [3]:
def verify_json(schema=schema_file, data=data_file, max_depth = None):
    engine = Engine(schema=schema, target=data, max_depth=max_depth)
    log = engine.run()
    return log

In [4]:
# valid json data verification
verify_json(data=data_file)

--- Validation Error Log ---

(maximum stack depth: 3)
--- validation done ---
1


'valid'

In [5]:
# type error verification
log = verify_json(data=data_type_error_file)
target_errors = {
    'top_object.a': ValidationError(ErrorType.BAD_VALUE, {'value': 'c', 'rule': 'type(number)'}),
    'top_object': ValidationError(ErrorType.INCOMPLETE, {'value': 'a'})
}

# print(log)
# assert_all_logs(log, target_errors)
log

--- Validation Error Log ---

invalid json item: path(top_object.a), value(Value[c])
  schema id: 2
    <BAD VALUE>: value(c) violates schema[type(number)]

invalid json item: path(top_object), value(Object[a, b, d])
  schema id: 0
    <INCOMPLETE>: value({'a': 'c', 'b': 2834964989560816147, 'd': 1079245023883434373}) violates schema[properties], schema id: 1

(maximum stack depth: 3)
--- validation done ---
1


'invalid'

In [6]:
# unclosed error verification
log = verify_json(data=data_unclosed_error_file)

target_errors = {
    'top_object.d': ValidationError(ErrorType.UNCLOSED),
    'top_object': ValidationError(ErrorType.UNCLOSED)
}

# assert_all_logs(log, target_errors)
log

--- Validation Error Log ---

invalid json item: path(top_object.d), value(Array[3])
    <UNCLOSED>: unclosed structure

invalid json item: path(top_object), value(Object[a, b, d])
    <UNCLOSED>: unclosed structure

(maximum stack depth: 3)
--- validation done ---
1


'invalid'

In [ ]:
# maximum stack depth verification
log = verify_json(data=data_file, max_depth=2)

target_errors = {
    'circuit_breaker': ValidationError(ErrorType.DEPTH_ERROR, {'depth': 2})
}
# assert_all_logs(log, target_errors)
log

--- Validation Error Log ---

invalid json item: path(circuit_breaker), value(None)
    <DEPTH ERROR>: maximum allowed depth exceeded: 2

(maximum stack depth: 2)
--- validation done ---
0
enter


In [8]:
# enumeration error verification
log = verify_json(data=data_enum_error_file)
target_errors = {
    'top_object.d[2]': ValidationError(ErrorType.BAD_VALUE, {'value': 5, 'rule': 'enum([3, 4])'}),
    'top_object.d': ValidationError(ErrorType.INCOMPLETE, {'value': 2}),
    'top_object': ValidationError(ErrorType.INCOMPLETE, {'value': 'd'})
}

# assert_all_logs(log, target_errors)
log

--- Validation Error Log ---

invalid json item: path(top_object.d[2]), value(Value[5])
  schema id: 7
    <BAD VALUE>: value(5) violates schema[enum([3, 4])]

invalid json item: path(top_object.d), value(Array[3, 4, 5])
  schema id: 6
    <INCOMPLETE>: value([3, 4, 5]) violates schema[items], schema id: 7

invalid json item: path(top_object), value(Object[a, b, d])
  schema id: 0
    <INCOMPLETE>: value({'a': 1, 'b': 2834964989560816147, 'd': 4003026094496801395}) violates schema[properties], schema id: 1

(maximum stack depth: 3)
--- validation done ---
1


'invalid'

In [9]:
# unexpected json object verification
log = verify_json(data=data_no_schema_error_file)

target_errors = {
    'top_object.b.e': ValidationError(ErrorType.UNEXPECTED, {'value': 'extra'}),
    'top_object.b': ValidationError(ErrorType.INCOMPLETE, {'value': 'e'}),
    'top_object': ValidationError(ErrorType.INCOMPLETE, {'value': 'b'})
}

# assert_all_logs(log, target_errors)
log

--- Validation Error Log ---

invalid json item: path(top_object.b.e), value(Value[extra])
  schema id: -2
    <UNEXPECTED>: unexpected: extra

invalid json item: path(top_object.b), value(Object[c, e])
  schema id: 3
    <INCOMPLETE>: value({'c': None, 'e': 'extra'}) violates schema[properties], schema id: 4

invalid json item: path(top_object), value(Object[a, b, d])
  schema id: 0
    <INCOMPLETE>: value({'a': 1, 'b': -8322098377440451230, 'd': 1079245023883434373}) violates schema[properties], schema id: 1

(maximum stack depth: 3)
--- validation done ---
1


'invalid'